# dispatcher-watts — v1 results

Backtest of battery dispatch strategies against **2024 and 2025 ERCOT real-time
settlement-point prices** (15-minute intervals, four trading hubs).

- **Battery:** 1 MWh / 0.5 MW (2-hour duration), 87% round-trip efficiency.
- **Strategies:** `threshold`, `rolling-average`, and a `perfect-foresight` LP
  benchmark (the theoretical revenue ceiling).

Run top to bottom. It needs the eight cached hub-years; fetch any that are
missing with `dispatcher-watts data fetch --year <YEAR> --hub <HUB>`.

In [ ]:
import polars as pl

from dispatcher_watts.backtest.engine import run_backtest
from dispatcher_watts.backtest.metrics import capture_rate, compute_metrics
from dispatcher_watts.battery.model import Battery, BatterySpec
from dispatcher_watts.data.schemas import ERCOT_HUBS
from dispatcher_watts.data.store import load_prices
from dispatcher_watts.reporting.charts import (
    cumulative_revenue_chart,
    daily_revenue_chart,
    dispatch_chart,
    soc_chart,
)
from dispatcher_watts.strategies.perfect_foresight import PerfectForesightStrategy
from dispatcher_watts.strategies.rolling_avg import RollingAverageStrategy
from dispatcher_watts.strategies.threshold import ThresholdStrategy

SPEC = BatterySpec()  # v1 reference battery: 1 MWh / 0.5 MW / 87% RTE
YEARS = (2024, 2025)
SPEC

## Backtest every strategy across all hubs and years

Each strategy is run on a fresh battery for all four hubs and both years. The
`perfect-foresight` LP for each hub-year is the ceiling its capture rate is
measured against.

In [ ]:
def strategies():
    """Fresh strategy instances for one backtest run."""
    return {
        "threshold": ThresholdStrategy(charge_below=20.0, discharge_above=50.0),
        "rolling-average": RollingAverageStrategy(window_hours=24.0),
        "perfect-foresight": PerfectForesightStrategy(SPEC),
    }


rows = []
for year in YEARS:
    for hub in ERCOT_HUBS:
        prices = load_prices(year, hub)
        metrics = {
            name: compute_metrics(run_backtest(prices, Battery(SPEC), strat))
            for name, strat in strategies().items()
        }
        ceiling = metrics["perfect-foresight"].total_revenue
        for name, m in metrics.items():
            rows.append(
                {
                    "year": year,
                    "hub": hub,
                    "strategy": name,
                    "revenue": round(m.total_revenue),
                    "capture_rate": round(capture_rate(m.total_revenue, ceiling), 3),
                    "cycles": round(m.equivalent_full_cycles),
                }
            )

results = pl.DataFrame(rows)
results

## Headline: revenue per MWh-year, by strategy

The battery is 1 MWh, so total revenue is already revenue per MWh-year.

In [ ]:
results.pivot(values="revenue", index=["year", "hub"], on="strategy")

## Capture rate

Capture rate is strategy revenue divided by the perfect-foresight ceiling.
Real operators typically capture 60–80%. These two rule-based strategies use
untuned default parameters and fall well short — shown honestly rather than
tuned to flatter the result.

In [ ]:
(
    results.filter(pl.col("strategy") != "perfect-foresight")
    .group_by("strategy")
    .agg(pl.col("capture_rate").mean().round(3).alias("mean_capture_rate"))
    .sort("mean_capture_rate", descending=True)
)

## Year over year: arbitrage spreads compressed

The perfect-foresight benchmark is the cleanest measure of how much arbitrage
value the market actually offered. It fell at every hub from 2024 to 2025 —
consistent with growing battery saturation eroding price spreads.

In [ ]:
perfect = results.filter(pl.col("strategy") == "perfect-foresight")
spread = perfect.pivot(values="revenue", index="hub", on="year")
spread.with_columns(
    ((pl.col("2025") - pl.col("2024")) / pl.col("2024") * 100).round(1).alias("change_%")
)

## Dispatch behaviour — HB_HOUSTON 2024

Charts for one representative hub-year. `perfect-foresight` shows what optimal
dispatch looks like; `threshold` shows a simple rule in action.

In [ ]:
prices = load_prices(2024, "HB_HOUSTON")
threshold_run = run_backtest(prices, Battery(SPEC), ThresholdStrategy(20.0, 50.0))
perfect_run = run_backtest(prices, Battery(SPEC), PerfectForesightStrategy(SPEC))

dispatch_chart(perfect_run).show()

In [ ]:
soc_chart(perfect_run).show()

In [ ]:
cumulative_revenue_chart(threshold_run).show()
cumulative_revenue_chart(perfect_run).show()

In [ ]:
daily_revenue_chart(perfect_run).show()

## Honest limitations

- **Energy arbitrage only.** Real operators also earn ancillary-services
  revenue (frequency regulation, reserves), often more than half the total.
- **The backtest overstates real revenue.** It assumes no dispatch latency,
  no forecast error, and no outages.
- **Perfect foresight is a ceiling, not a strategy** — it cannot be deployed.
- **No cherry-picking.** Every hub and both years are shown, good and bad.